In [0]:
%python
salary_df = spark.sql("""
SELECT *
FROM hr_poc.silver.salary
""")

display(salary_df)

In [0]:
-- %python
-- spark.sql("""
-- CREATE TABLE IF NOT EXISTS hr_poc.gold.fact_salary
-- USING DELTA
-- AS
-- SELECT
--     employee_id,
--     effective_date,
--     salary,
--     bonus,
--     salary_grade
-- FROM hr_poc.silver.salary
-- """)

-- spark.sql("""
-- CREATE TABLE IF NOT EXISTS hr_poc.gold.fact_attendance
-- USING DELTA
-- AS
-- SELECT
--     employee_id,
--     attendance_date,
--     attendance_status,
--     working_hours,
--     overtime_hours
-- FROM hr_poc.silver.attendance
-- """)

-- spark.sql("""
-- CREATE TABLE IF NOT EXISTS hr_poc.gold.fact_attrition
-- USING DELTA
-- AS
-- SELECT
--     employee_id,
--     exit_date,
--     exit_reason,
--     exit_type
-- FROM hr_poc.silver.attrition
-- """)

In [0]:
%python
salary_new_df = spark.sql("SELECT * FROM hr_poc.silver.salary")
salary_new_df.show()
(
    salary_new_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("hr_poc.gold.fact_salary")
)

attrition_new_df = spark.sql("SELECT * FROM hr_poc.silver.attrition")
attrition_new_df.show()
(
    attrition_new_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("hr_poc.gold.fact_attrition")
)
       
attendance_new_df = spark.sql("SELECT * FROM hr_poc.silver.attendance")
attendance_new_df.count()
(
    attendance_new_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("hr_poc.gold.fact_attendance")
)

In [0]:
%python
display(spark.sql("""
SELECT a.employee_id
FROM hr_poc.silver.attrition a
LEFT JOIN hr_poc.silver.fact_employee e
    ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL
"""))

In [0]:
%sql
SELECT
    f.employee_id,
    d.employee_name,
    d.department_id,
    d.Designation_id,
    d.location,
    f.salary,
    d.is_current
FROM hr_poc.gold.fact_salary f
JOIN hr_poc.gold.fact_employee d
    ON f.employee_id = d.employee_id
WHERE d.is_current = 'Y';


-- display(spark.sql("""
-- SELECT
--     a.employee_id,
--     e.employee_name,
--     e.department_id,
--     e.location,
--     a.attendance_date,
--     a.attendance_status,
--     a.working_hours,
--     a.overtime_hours
-- FROM hr_poc.gold.fact_attendance a
-- JOIN hr_poc.gold.fact_employee e
--     ON a.employee_id = e.employee_id
-- WHERE e.is_current = 'Y'
-- LIMIT 20
-- """))

In [0]:
%python
display(spark.sql("""
SELECT
    a.employee_id,
    e.employee_name,
    e.department_id,
    e.location,
    a.attendance_date,
    a.attendance_status,
    a.working_hours,
    a.overtime_hours
FROM hr_poc.gold.fact_attendance a
JOIN hr_poc.gold.fact_employee e
    ON a.employee_id = e.employee_id
WHERE e.is_current = 'Y'
LIMIT 20
"""))

In [0]:
%python
spark.sql("""
CREATE TABLE IF NOT EXISTS hr_poc.gold.dim_date
USING DELTA
AS
SELECT
    date_format(date, 'yyyyMMdd') AS date_key,
    date,
    year(date) AS year,
    quarter(date) AS quarter,
    month(date) AS month,
    date_format(date, 'MMMM') AS month_name,
    weekofyear(date) AS week_of_year,
    day(date) AS day,
    date_format(date, 'EEEE') AS day_name
FROM (
    SELECT explode(
        sequence(
            to_date('2020-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )
    ) AS date
)
""")


display(spark.sql("""
SELECT *
FROM hr_poc.gold.dim_date
ORDER BY date
LIMIT 2
"""))

In [0]:
%python
spark.sql("""
SELECT COUNT(*) AS total_dates
FROM hr_poc.gold.dim_date
""").show()

In [0]:
%python
# spark.sql("""
# ALTER TABLE hr_poc.gold.fact_attendance
# ADD COLUMNS (
#     date_key STRING
# )
# """)
spark.sql("""
UPDATE hr_poc.gold.fact_attendance
SET date_key = date_format(attendance_date, 'yyyyMMdd')
""")

display(spark.sql("""
SELECT
    employee_id,
    attendance_date,
    date_key,
    attendance_status,
    working_hours
FROM hr_poc.gold.fact_attendance
LIMIT 3
"""))

In [0]:
%python
# spark.sql("""
# ALTER TABLE hr_poc.gold.fact_salary
# ADD COLUMNS (
#     date_key STRING
# )
# """)

spark.sql("""
UPDATE hr_poc.gold.fact_salary
SET date_key = date_format(effective_date, 'yyyyMMdd')
""")

display(spark.sql("""
SELECT
    employee_id,
    effective_date,
    date_key,
    salary,
    bonus
FROM hr_poc.gold.fact_salary
LIMIT 3
"""))

In [0]:
%python
# spark.sql("""
# ALTER TABLE hr_poc.gold.fact_attrition
# ADD COLUMNS (
#     date_key STRING
# )
# """)

spark.sql("""
UPDATE hr_poc.gold.fact_attrition
SET date_key = date_format(exit_date, 'yyyyMMdd')
""")

display(spark.sql("""
SELECT
    employee_id,
    exit_date,
    date_key,
    exit_reason,
    exit_type
FROM hr_poc.gold.fact_attrition
LIMIT 10
"""))

In [0]:
%python
display(spark.sql("""
SELECT
    d.department_id, d.department_name,
    COUNT(DISTINCT employee_id) AS employee_count
FROM hr_poc.gold.fact_employee
left JOIN hr_poc.silver.department d
    ON fact_employee.department_id = d.department_id
WHERE is_current = 'Y'
GROUP BY d.department_id, d.department_name
ORDER BY employee_count DESC
"""))

In [0]:
%python
display(spark.sql("""
SELECT
    e.department_id,
    COUNT(DISTINCT s.employee_id) AS employees,
    SUM(s.salary) AS total_salary,
    AVG(s.salary) AS average_salary
FROM hr_poc.gold.fact_salary s
JOIN hr_poc.gold.fact_employee e
    ON s.employee_id = e.employee_id
WHERE e.is_current = 'Y'
GROUP BY e.department_id
ORDER BY total_salary DESC
"""))

In [0]:
%python
display(spark.sql("""
SELECT
    a.employee_id,
    a.date_key,
    d.year,
    d.month_name,
    a.attendance_status,
    COUNT(*) AS attendance_count,
    AVG(a.working_hours) AS avg_working_hours
FROM hr_poc.gold.fact_attendance a
JOIN hr_poc.gold.dim_date d
    ON a.date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name,
    a.attendance_status,
    a.employee_id,
    a.date_key
ORDER BY
    a.employee_id,
    a.date_key
"""))

In [0]:
%python
print("Salary:",
      spark.table("hr_poc.gold.fact_salary").count())

print("Attendance:",
      spark.table("hr_poc.gold.fact_attendance").count())

print("Attrition:",
      spark.table("hr_poc.gold.fact_attrition").count())

print("Employee:",
      spark.table("hr_poc.gold.fact_employee").count())

In [0]:
%python
display(spark.sql("select  * from hr_poc.gold.fact_salary  order by 1 desc"))